# 核心研究区选取 与 轨迹点筛选 —— 代码整理

背景：ArcGIS 的时空立方体(Gi*+Mann-Kendall)方法在这份数据上跑不出可用结果
(全局均值/标准差被 GPS 漂移点污染，z 分数全部趋近于 0)，所以改用下面这条更
简单、天与天之间不共享任何统计量的路线。

**分四部分：**

0. 核密度估计(KDE) —— 每天的 GPS 点云 → 一张密度栅格(kd_baseline_all.tif)
1. 持续性热点选取 —— 7 天密度栅格 → 0~7 的"持续性计数"栅格
2. 候选区聚类与清单 —— 连通域聚类 → 128 个候选研究区(面积/行政区/中心点)
3. 轨迹点筛选 —— 用候选区裁剪 GPS 轨迹，保持轨迹连续性不被切碎

**依赖环境**：`numpy` `pandas` `geopandas` `rasterio` `shapely>=2.0` `pyproj`。
**不用 scipy**（本机 scipy 装不上，pip 哈希校验一直失败），凡是原本会用
`scipy.ndimage.label` 的地方都改成了手写的 numpy/Python 连通域算法。

以下代码块是从生产脚本 `18_persistent_hotspot_select.py` /
`19_study_area_report.py` / `20_filter_points_by_study_area.py`
整理出来的核心逻辑，路径都是占位符，**用于阅读代码逻辑，不是拿来在 Colab
里直接跑**（原始数据是本地 7.5GB+ 的 GPS 轨迹文件，没有上传）。

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import shapes as rio_shapes
import geopandas as gpd
import shapely
from shapely import STRtree, points as shapely_points
from pyproj import Transformer

## 第零部分：核密度估计(KDE) —— 生成每天的密度栅格

这是最开始的一步，7天里每一天都单独跑一次，把当天全部GPS点的空间分布
算成一张连续的密度栅格(`kd_baseline_all.tif`)，后面第一部分的"持续性热点"
就是拿这7张栅格去做跨天统计。

跟ArcGIS Pro自带的Kernel Density工具对标（这台机器没有Spatial Analyst许可，
所以自己实现），核心思路一致：

1. **点先落网格**：把点(转成UTM米制坐标后)累加成一张细网格的二维直方图。
   因为数据量大(单天可能几千万行)，用 `pandas.read_csv(chunksize=...)`
   分块读取、增量累加直方图——直方图满足线性可加性，不需要一次性把全部
   点读进内存。
2. **卷积**：直方图算完后，跟一个**quartic(双权重)核函数**做一次卷积，
   把"点的计数"变成"连续的密度场"。核函数公式：

   $$k(d) = \frac{3}{\pi r^2}\left(1 - (d/r)^2\right)^2, \quad d < r$$

   卷积用FFT实现（数学上跟直接卷积等价，但对几千x几千的网格快得多）。
3. **筛点**：只保留落在北京市行政边界内的点（`boundary_utils.beijing_mask`，
   用省级行政区划矢量做点在多边形判断，比粗糙的经纬度矩形框准确——矩形框
   会把跑到河北/天津但仍在矩形内的点也算进来）。

几个容易踩坑、写代码时特意处理过的细节：

- **南北方向**：`numpy.histogram2d(y, x, ...)` 按 y 递增排行（行索引0=南边），
  但 GeoTIFF 的"北向上"约定是行索引0=北边，两者顺序相反。不翻转的话整张图
  会南北镜像颠倒（不是平移偏移，是整体上下翻转）——第一版就在这里栽过一次，
  后来加了 `np.flipud()` 修好。
- **FFT卷积的浮点噪声**：理论上离任何点超过 `search_radius` 的像元密度该精确
  为0，但FFT卷积会在这些像元上留下量级 1e-16~1e-24 的浮点噪声。用默认拉伸
  渲染时这些噪声会被当成"极小的真实密度值"，铺满全图变成噪点，跟真正的
  稀疏区域没法区分。所以换算单位前先按峰值的千万分之一做阈值，把噪声钉死
  成精确的0（跟GeoTIFF的nodata=0.0对得上）。
- **单位换算**：核函数公式按"每平方米"归一化，换算成"每平方公里"（对应
  arcpy版 `area_unit_scale_factor="SQUARE_KILOMETERS"`），量级才好读。

In [ ]:
def quartic_kernel(radius_cells):
    """离散化的quartic(双权重)核，中心归一化到能直接乘计数值"""
    r = radius_cells
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    dist = np.sqrt(xx.astype(np.float64) ** 2 + yy.astype(np.float64) ** 2)
    k = np.zeros_like(dist)
    mask = dist < r
    k[mask] = (1 - (dist[mask] / r) ** 2) ** 2
    k *= 3.0 / (np.pi * r ** 2)
    return k


def _next_pow2(n):
    """不依赖scipy.fft.next_fast_len，退而求其次用2的下一个幂次
    (numpy.fft原生支持，没有scipy也能跑)"""
    p = 1
    while p < n:
        p *= 2
    return p

In [ ]:
class HistogramAccumulator:
    """先把点(UTM米制坐标)累加成细网格2D直方图，读完所有数据分块后再一次性
    做FFT核卷积。cell_size/search_radius语义跟ArcGIS Kernel Density一致。"""

    def __init__(self, extent_utm, cell_size=150.0, search_radius=800.0):
        self.cell_size = cell_size
        self.search_radius = search_radius
        xmin, ymin, xmax, ymax = extent_utm
        pad = search_radius  # 缓冲圈，让研究区边界外的点也能贡献边界像元密度
        self.xmin_p, self.ymin_p = xmin - pad, ymin - pad
        self.xmax_p, self.ymax_p = xmax + pad, ymax + pad
        self.xmin, self.ymin, self.xmax, self.ymax = xmin, ymin, xmax, ymax
        self.n_cols = int(np.ceil((self.xmax_p - self.xmin_p) / cell_size))
        self.n_rows = int(np.ceil((self.ymax_p - self.ymin_p) / cell_size))
        self.hist = np.zeros((self.n_rows, self.n_cols), dtype=np.float64)
        self.n_points_added = 0

    def add(self, x, y):
        """每个分块调用一次，增量累加到同一张直方图上"""
        x, y = np.asarray(x, dtype=np.float64), np.asarray(y, dtype=np.float64)
        valid = (np.isfinite(x) & np.isfinite(y) &
                 (x >= self.xmin_p) & (x < self.xmax_p) &
                 (y >= self.ymin_p) & (y < self.ymax_p))
        if not valid.any():
            return
        x, y = x[valid], y[valid]
        h, _, _ = np.histogram2d(
            y, x, bins=[self.n_rows, self.n_cols],
            range=[[self.ymin_p, self.ymax_p], [self.xmin_p, self.xmax_p]],
        )
        self.hist += h
        self.n_points_added += len(x)

    def finalize(self, out_path=None):
        """做FFT核卷积、裁掉缓冲圈、翻转到北向上、清理浮点噪声、
        换算成'每平方公里'密度，可选写出GeoTIFF"""
        radius_cells = max(1, int(round(self.search_radius / self.cell_size)))
        kernel = quartic_kernel(radius_cells)

        fshape = [self.hist.shape[i] + kernel.shape[i] - 1 for i in range(2)]
        fast_len = [_next_pow2(s) for s in fshape]
        H = np.fft.rfft2(self.hist, fast_len)
        K = np.fft.rfft2(kernel, fast_len)
        conv = np.fft.irfft2(H * K, fast_len)[:fshape[0], :fshape[1]]
        start = [(kernel.shape[i] - 1) // 2 for i in range(2)]
        density = conv[start[0]:start[0] + self.hist.shape[0], start[1]:start[1] + self.hist.shape[1]]

        row0 = int(round((self.ymax_p - self.ymax) / self.cell_size))
        row1 = int(round((self.ymax_p - self.ymin) / self.cell_size))
        col0 = int(round((self.xmin - self.xmin_p) / self.cell_size))
        col1 = int(round((self.xmax - self.xmin_p) / self.cell_size))
        density = density[row0:row1, col0:col1]

        # histogram2d按y递增排行(行0=南边)，GeoTIFF北向上约定行0=北边，翻转对齐
        density = np.flipud(density)

        # FFT卷积在"理论上该精确为0"的像元上留下的极小浮点噪声，钉死成0
        noise_floor = density.max() * 1e-6 if density.max() > 0 else 0.0
        density[density < noise_floor] = 0.0

        # 换算成"每平方公里"密度(对应area_unit_scale_factor="SQUARE_KILOMETERS")
        density = np.clip(density * (1000.0 / self.cell_size) ** 2, 0, None).astype(np.float32)

        if out_path:
            from rasterio.transform import Affine
            transform = Affine(self.cell_size, 0, self.xmin, 0, -self.cell_size, self.ymax)
            with rasterio.open(
                out_path, "w", driver="GTiff",
                height=density.shape[0], width=density.shape[1], count=1,
                dtype=density.dtype, crs="EPSG:32650", transform=transform,
                compress="lzw", nodata=0.0,
            ) as dst:
                dst.write(density, 1)
        return density, out_path

In [ ]:
# 示例调用（对某一天的<日期>_core_features.csv跑一次）—— 不在此notebook里实际执行
# from boundary_utils import beijing_mask
#
# def lonlat_to_utm(lon, lat):
#     from pyproj import Transformer
#     return Transformer.from_crs("EPSG:4326", "EPSG:32650", always_xy=True).transform(lon, lat)
#
# extent_utm = (min_x, min_y, max_x, max_y)  # 研究区范围(UTM)，7天固定用同一个extent才能对齐
# acc = HistogramAccumulator(extent_utm, cell_size=150.0, search_radius=800.0)
# for chunk in pd.read_csv("20170301_core_features.csv", usecols=["latitude", "longitude"], chunksize=2_000_000):
#     chunk = chunk[beijing_mask(chunk["longitude"].values, chunk["latitude"].values)]
#     x, y = lonlat_to_utm(chunk["longitude"].values, chunk["latitude"].values)
#     acc.add(x, y)
# density, path = acc.finalize("20170301/kd_baseline_all.tif")

这一步单天要跑几千万行GPS点，7天各跑一次，输出7张 `kd_baseline_all.tif`——
就是下面第一部分 `build_persistence_raster()` 的输入。

## 第一部分：持续性热点选取

对每一天的 KDE 密度栅格，**只在当天内部**算百分位排名，取密度最高的
`top_pct`（默认 3%）标记为"当天热点"。7 天各自独立打标之后，逐像元累加
"这个像元 7 天里有几天是热点"，得到一个 0~7 的持续性计数栅格。

跟 Gi* 的区别：每天的阈值只用当天自己的分位数计算，天与天之间不共享
任何全局统计量——某一天数据有问题也只会让那一天跑偏，不会传染给其它天。

In [ ]:
def build_persistence_raster(day_tif_paths: dict, top_pct: float = 3.0, out_path: str = None):
    """day_tif_paths: {"20170301": "path/to/day1/kd_baseline_all.tif", ...} 共7天
    返回 (persistence, mean_norm, profile)，并可选写出2波段GeoTIFF"""
    ref_profile = None
    daily_masks, daily_norm = [], []

    for label, path in day_tif_paths.items():
        with rasterio.open(path) as ds:
            arr = ds.read(1).astype(np.float64)
            profile = ds.profile
        if ref_profile is None:
            ref_profile = profile

        valid = arr > 0
        threshold = np.percentile(arr[valid], 100.0 - top_pct)
        mask = (arr >= threshold) & valid
        daily_masks.append(mask)

        vmax = arr[valid].max()
        daily_norm.append(np.where(valid, arr / vmax, 0.0))

        print(f"[{label}] 有效像元={int(valid.sum()):,}, top{top_pct:.0f}%阈值={threshold:.2f}, "
              f"当天热点像元数={int(mask.sum()):,}")

    n_days = len(day_tif_paths)
    persistence = np.zeros_like(daily_masks[0], dtype=np.int16)
    for m in daily_masks:
        persistence += m.astype(np.int16)
    mean_norm = np.mean(np.stack(daily_norm, axis=0), axis=0).astype(np.float32)

    if out_path:
        profile = ref_profile.copy()
        profile.update(count=2, dtype="float32", nodata=-1.0, compress="lzw")
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(persistence.astype(np.float32), 1)
            dst.write(mean_norm, 2)
            dst.set_band_description(1, f"persistence_count_of_{n_days}_days_top{top_pct:.0f}pct")
            dst.set_band_description(2, "mean_normalized_density_across_days")

    return persistence, mean_norm, ref_profile

In [ ]:
# 示例调用（路径按你本地7天KDE输出结构填）——不在此notebook里实际执行
# day_paths = {
#     "20170301": "E:/.../02-07/vm_seven_day_results_.../outputs/20170301/kd_baseline_all.tif",
#     "20170302": "...",
#     ...  # 共7天
# }
# persistence, mean_norm, profile = build_persistence_raster(
#     day_paths, top_pct=3.0, out_path="study_area_candidates.tif"
# )

**实际跑出的结果**（`top_pct=3`，7天）：persistence 值分布（像元数）：

| persistence | 像元数 |
|---|---|
| 7（7天全是热点） | 6,663 |
| 6 | 778 |
| 5 | 838 |
| 4 | 530 |
| 3 | 581 |
| 2 | 1,169 |
| 1 | 1,307 |
| 0 | 624,682 |

后续只取 `persistence == 7` 的像元进入第二部分聚类。

## 第二部分：候选区聚类与清单

对 `persistence >= min_count`（这里用 7，即7天全部是热点）的像元做 **8邻域
连通域分析**，一片连通区域就是一个候选研究片区，小于 `min_cells` 的碎片丢弃。
再算每个候选区的面积、几何中心（反投影回经纬度）、落在北京哪个区。

连通域标记不用 `scipy.ndimage.label`（本机装不上），改用纯 numpy/Python 的
BFS flood-fill——待标记像元数量远小于栅格总像元数，直接 BFS 足够快。

In [ ]:
_NEIGHBOR_OFFSETS_8 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]

def label_connected_components(mask):
    """手写8邻域连通域标记，替代scipy.ndimage.label"""
    h, w = mask.shape
    labels = np.zeros((h, w), dtype=np.int32)
    current_label = 0
    ys, xs = np.where(mask)
    for y0, x0 in zip(ys, xs):
        if labels[y0, x0] != 0:
            continue
        current_label += 1
        stack = [(y0, x0)]
        labels[y0, x0] = current_label
        while stack:
            y, x = stack.pop()
            for dy, dx in _NEIGHBOR_OFFSETS_8:
                ny, nx = y + dy, x + dx
                if 0 <= ny < h and 0 <= nx < w and mask[ny, nx] and labels[ny, nx] == 0:
                    labels[ny, nx] = current_label
                    stack.append((ny, nx))
    return labels, current_label

In [ ]:
def build_candidate_regions(persistence, mean_norm, profile, min_count=7, min_cells=3):
    """assign_beijing_district 来自 boundary_utils.py，依赖北京行政区划shp文件，
    这里只给出调用方式，具体实现见 boundary_utils.load_beijing_districts()/assign_beijing_district()"""
    from boundary_utils import assign_beijing_district  # 项目内的行政区划判断工具

    transform, crs = profile["transform"], profile["crs"]
    cell_area_km2 = abs(transform.a * transform.e) / 1e6

    mask = persistence >= min_count
    labeled, n_clusters = label_connected_components(mask)
    print(f"min_count>={min_count} 的像元共 {int(mask.sum()):,} 个, 初步连通域 {n_clusters} 个")

    to_wgs84 = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
    rows, geoms = [], []

    for cluster_id in range(1, n_clusters + 1):
        cluster_mask = labeled == cluster_id
        n_cells = int(cluster_mask.sum())
        if n_cells < min_cells:
            continue

        ys, xs = np.where(cluster_mask)
        xs_map = transform.c + (xs + 0.5) * transform.a
        ys_map = transform.f + (ys + 0.5) * transform.e
        cx_lon, cy_lat = to_wgs84.transform(xs_map.mean(), ys_map.mean())

        rows.append({
            "n_cells": n_cells,
            "area_km2": round(n_cells * cell_area_km2, 3),
            "mean_persistence": round(float(persistence[cluster_mask].mean()), 2),
            "mean_density_norm": round(float(mean_norm[cluster_mask].mean()), 3),
            "center_lon": round(cx_lon, 5),
            "center_lat": round(cy_lat, 5),
        })
        cluster_arr = cluster_mask.astype(np.uint8)
        polys = [shapely.geometry.shape(g) for g, v in rio_shapes(cluster_arr, transform=transform) if v == 1]
        geoms.append(shapely.unary_union(polys))

    order = sorted(range(len(rows)), key=lambda i: -rows[i]["area_km2"])
    rows = [rows[i] for i in order]
    geoms = [geoms[i] for i in order]

    districts = assign_beijing_district(
        np.array([r["center_lon"] for r in rows]),
        np.array([r["center_lat"] for r in rows]),
    )
    for i, (r, d) in enumerate(zip(rows, districts), 1):
        r["district"] = d
        r["rank"] = i

    gdf = gpd.GeoDataFrame(rows, geometry=geoms, crs=crs)
    return gdf[["rank", "district", "area_km2", "n_cells", "mean_persistence",
                "mean_density_norm", "center_lon", "center_lat", "geometry"]]

In [ ]:
# 示例调用 —— 不在此notebook里实际执行
# gdf = build_candidate_regions(persistence, mean_norm, profile, min_count=7, min_cells=3)
# gdf.to_file("study_area_candidates.gpkg", driver="GPKG")
# gdf.drop(columns="geometry").to_csv("study_area_candidates.csv", index=False, encoding="utf-8-sig")

**实际跑出的结果**：`min_count=7, min_cells=3` → 128 个候选区。前几名：

| rank | district | area_km2 | mean_persistence | center_lon | center_lat |
|---|---|---|---|---|---|
| 1 | 东城区 | 100.102 | 7.0 | 116.41591 | 39.92185 |
| 2 | 朝阳区 | 2.655 | 7.0 | 116.41191 | 39.99132 |
| 3 | 海淀区 | 2.542 | 7.0 | 116.31258 | 39.93412 |
| 4 | 顺义区 | 2.475 | 7.0 | 116.61207 | 40.04726 |
| 5 | 朝阳区 | 2.025 | 7.0 | 116.58376 | 40.07167 |

一个 100 km² 的中心城区大核心区 + 127 个更小的分散次级候选区（完整清单见
`study_area_candidates.csv`）。

## 第三部分：轨迹点筛选（保持轨迹连续性）

用第二部分产出的 128 个候选区多边形筛选某一天的 GPS 轨迹点
（`01_feature_engineering.py` 的输出 `<日期>_core_features.csv`）。

跟"直接丢弃候选区外的点"不一样，这里**保留轨迹连续性**：以已有的
`trip_break` 字段分段（同一辆车相邻两点间隔超过 `max_gap_sec` 就算新的一段）
——一段路段只要有任意一个点落在候选区里，就把这一整段（含区外的点）原样保留。
这样同一辆车的一段轨迹在时间上还是连续的，能看出车辆进出候选区前后的完整
运动过程，不会被简单空间裁剪切成一堆看不出运动方向的孤立点。

空间判定用 shapely 2.0 的 `STRtree` 做批量查询——128个候选区加起来占北京
总面积不到1%，STRtree 的包围盒剪枝比逐个候选区做全量 `contains` 快得多。

数据本身按 `taxi_id + timestamp` 全局排序，所以按 taxi 分块流式处理即可，
不需要把全部数据一次性读入内存。

In [ ]:
def load_regions(gpkg_path):
    gdf = gpd.read_file(gpkg_path).sort_values("rank").reset_index(drop=True)
    tree = STRtree(gdf.geometry.values)
    ranks = gdf["rank"].values.astype(np.int32)
    return tree, ranks

def assign_candidate_rank(x, y, tree, ranks):
    """x, y: UTM米制坐标(不是经纬度)。返回每个点落在哪个候选区的rank，不在任何候选区内为-1"""
    pts = shapely_points(x, y)
    query_idx, tree_idx = tree.query(pts, predicate="intersects")
    result = np.full(len(x), -1, dtype=np.int32)
    result[query_idx] = ranks[tree_idx]
    return result

def flush_taxi_block(rows_df, out_columns):
    """rows_df: 单个taxi_id、按timestamp排序的完整数据块。
    按trip_break切分连续路段，逐段判断是否命中候选区，命中就整段保留。"""
    trip_break = rows_df["trip_break"].values.astype(bool)
    seg_id = np.cumsum(trip_break) - 1
    in_area = rows_df["candidate_rank"].values >= 0

    keep_mask = np.zeros(len(rows_df), dtype=bool)
    for sid in np.unique(seg_id):
        sel = seg_id == sid
        if in_area[sel].any():
            keep_mask |= sel

    return rows_df.loc[keep_mask, out_columns]

In [ ]:
def filter_points_by_study_area(input_csv, gpkg_path, output_csv, lonlat_to_utm, dtypes, chunk_size=2_000_000):
    """lonlat_to_utm: 经纬度(WGS84)转UTM米制坐标的函数，来自 kde_utils.py"""
    tree, ranks = load_regions(gpkg_path)
    out_columns = list(dtypes) + ["candidate_rank"]

    header_written = False
    total_in = total_kept = 0
    block_taxi, block_rows = None, []

    with open(output_csv, "w", newline="", encoding="utf-8-sig") as out_f:
        for chunk in pd.read_csv(input_csv, dtype=dtypes, chunksize=chunk_size, keep_default_na=False):
            x, y = lonlat_to_utm(chunk["longitude"].values, chunk["latitude"].values)
            chunk["candidate_rank"] = assign_candidate_rank(x, y, tree, ranks)

            for taxi_id, group in chunk.groupby("taxi_id", sort=False):
                if block_taxi is not None and taxi_id != block_taxi:
                    block_df = pd.concat(block_rows, ignore_index=True) if len(block_rows) > 1 else block_rows[0]
                    kept = flush_taxi_block(block_df, out_columns)
                    total_in += len(block_df); total_kept += len(kept)
                    if len(kept):
                        kept.to_csv(out_f, mode="a", header=not header_written, index=False)
                        header_written = True
                    block_rows = []
                block_taxi = taxi_id
                block_rows.append(group)

        if block_rows:
            block_df = pd.concat(block_rows, ignore_index=True) if len(block_rows) > 1 else block_rows[0]
            kept = flush_taxi_block(block_df, out_columns)
            total_in += len(block_df); total_kept += len(kept)
            if len(kept):
                kept.to_csv(out_f, mode="a", header=not header_written, index=False)

    print(f"输入 {total_in:,} 行, 保留(整段路段，含段内区外点) {total_kept:,} 行 "
          f"({total_kept/max(total_in,1)*100:.2f}%)")
    print("candidate_rank>=0 才是真正落在候选区内的点，-1是'同段有其它点命中候选区"
          "而被一并保留的区外点'，要严格空间子集的话额外筛 candidate_rank>=0 即可")

In [ ]:
# 示例调用 —— 不在此notebook里实际执行
# from kde_utils import lonlat_to_utm
# DTYPES = {
#     "group_id": "object", "taxi_id": "object", "timestamp": "int64",
#     "latitude": "float64", "longitude": "float64", "direction": "object",
#     "occupied": "int8", "positioning_valid": "int8",
#     "dt_sec": "object", "speed_gps_kmh": "object", "speed_reported_kmh": "object",
#     "heading_delta_deg": "object", "trip_break": "int8", "bad_coord": "int8",
#     "out_of_day": "int8", "event_type": "object", "is_outlier_drift": "int8",
# }
# filter_points_by_study_area(
#     "20170301_core_features.csv", "study_area_candidates.gpkg",
#     "20170301_filtered_by_study_area.csv", lonlat_to_utm, DTYPES,
# )

**目前进度**：只在200万行的截断样本上做过冒烟测试——保留 1,880,703 行
（94.04%），耗时约27.6秒。**7天完整数据的正式跑批还没做**，跑完之后每天会
产出一个 `<日期>_filtered_by_study_area.csv`，供后续针对这128个候选区做更
细的分析。